# Country Club Case Study - Tier 2

## Part 2: SQLite

Kirti Sharma

For this part of the assignment, I connect to the supplied SQLite database and answer Questions 10-13.

The database has three tables:
- Bookings
- Facilities
- Members

I used pandas only to display the query results.


In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("sqlite_db_pythonsqlite.db")

print("Connected to the Country Club database.")

Connected to the Country Club database.


### Check the tables

Before answering the questions, I will confirm the tables in the database.


In [2]:
pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

,name
0,Bookings
1,Facilities
2,Members


## Q10

Produce a list of facilities with total revenue less than 1000, sorted by revenue.

Guests and members have different prices, so I use a `CASE` statement to choose the correct cost. Revenue is the number of booked slots multiplied by the correct slot cost.


In [3]:
query_q10 = '''
SELECT
       f.name AS facility,
       SUM(
           CASE
               WHEN b.memid = 0 THEN b.slots * f.guestcost
               ELSE b.slots * f.membercost
           END
       ) AS total_revenue
FROM Facilities AS f
JOIN Bookings AS b
    ON f.facid = b.facid
GROUP BY f.facid, f.name
HAVING total_revenue < 1000
ORDER BY total_revenue;
'''

pd.read_sql_query(query_q10, conn)

,facility,total_revenue
0,Table Tennis,180
1,Snooker Table,240
2,Pool Table,270


## Q11

Produce a report of members and the member who recommended them.

I use a self-join because both the member and the recommender are stored in the `Members` table. A `LEFT JOIN` keeps members who do not have a recommender.


In [4]:
query_q11 = '''
SELECT
       m.surname,
       m.firstname,
       r.surname AS recommender_surname,
       r.firstname AS recommender_firstname
FROM Members AS m
LEFT JOIN Members AS r
    ON m.recommendedby = r.memid
WHERE m.memid <> 0
ORDER BY m.surname, m.firstname;
'''

pd.read_sql_query(query_q11, conn)

,surname,firstname,recommender_surname,recommender_firstname
0,Bader,Florence,Stibbons,Ponder
1,Baker,Anne,Stibbons,Ponder
2,Baker,Timothy,Farrell,Jemima
3,Boothe,Tim,Rownam,Tim
4,Butters,Gerald,Smith,Darren
5,Coplin,Joan,Baker,Timothy
6,Crumpet,Erica,Smith,Tracy
7,Dare,Nancy,Joplette,Janice
8,Farrell,David,None,None
9,Farrell,Jemima,None,None


## Q12

Find facility usage by member, excluding guests.

I use the total number of booked `slots` as the measure of facility usage because each booking can use more than one slot.


In [5]:
query_q12 = '''
SELECT
       f.name AS facility,
       m.firstname || ' ' || m.surname AS member,
       SUM(b.slots) AS total_slots
FROM Bookings AS b
JOIN Facilities AS f
    ON b.facid = f.facid
JOIN Members AS m
    ON b.memid = m.memid
WHERE b.memid <> 0
GROUP BY f.facid, f.name, m.memid, m.firstname, m.surname
ORDER BY f.name, m.surname, m.firstname;
'''

pd.read_sql_query(query_q12, conn)

,facility,member,total_slots
0,Badminton Court,Florence Bader,27
1,Badminton Court,Anne Baker,30
2,Badminton Court,Timothy Baker,21
3,Badminton Court,Tim Boothe,36
4,Badminton Court,Gerald Butters,63
...,...,...,...
197,Tennis Court 2,Darren Smith,57
198,Tennis Court 2,Jack Smith,3
199,Tennis Court 2,Tracy Smith,6
200,Tennis Court 2,Ponder Stibbons,96


## Q13

Find facility usage by month, excluding guests.

Again, I use total booked slots as the measure of usage. `strftime('%m', starttime)` extracts the month from the booking date.


In [6]:
query_q13 = '''
SELECT
       f.name AS facility,
       strftime('%m', b.starttime) AS month,
       SUM(b.slots) AS total_slots
FROM Bookings AS b
JOIN Facilities AS f
    ON b.facid = f.facid
WHERE b.memid <> 0
GROUP BY f.facid, f.name, strftime('%m', b.starttime)
ORDER BY month, f.name;
'''

pd.read_sql_query(query_q13, conn)

,facility,month,total_slots
0,Badminton Court,07,165
1,Massage Room 1,07,166
2,Massage Room 2,07,8
3,Pool Table,07,110
4,Snooker Table,07,140
5,Squash Court,07,50
6,Table Tennis,07,98
7,Tennis Court 1,07,201
8,Tennis Court 2,07,123
9,Badminton Court,08,414


## Summary

In this case study I practiced:
- filtering with `WHERE`;
- combining tables with `JOIN`;
- using `CASE` for different member and guest costs;
- grouping data with `GROUP BY`;
- filtering grouped results with `HAVING`;
- using a self-join for member recommendations;
- working with dates in SQLite;
- connecting Python to a SQLite database.


In [7]:
conn.close()
print("Database connection closed.")

Database connection closed.
